In [56]:
import json
import pathlib
import os
import sys  
sys.path.insert(0, '/nethome/bdevnani3/flash1/long_tail_lang/')
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm.notebook import trange, tqdm
from os import listdir
from os.path import isfile, join

In [57]:
from data_loader import dataloaders, classes
from clip import clip

In [58]:
dataset_path = '/nethome/bdevnani3/flash1/long_tail_lang/datasets/ImageNet/'
dataset = 'ImageNet_LT'
split = "train"

In [59]:
dl = dataloaders.load_data(data_root= dataset_path, dataset=dataset, phase=split, batch_size=1)

Loading data from /nethome/bdevnani3/flash1/long_tail_lang/data/ImageNet_LT/ImageNet_LT_train.txt
Use data transformation: Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.5, 1), ratio=(0.75, 1.3333), interpolation=bicubic)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=[0.6, 1.4], contrast=[0.6, 1.4], saturation=[0.6, 1.4], hue=None)
    ToTensor()
    Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
)
115846
No sampler.
Shuffle is True.


In [60]:
# Initialize CLIP models 
class TextEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.transformer = clip_model.transformer
        self.positional_embedding = clip_model.positional_embedding
        self.ln_final = clip_model.ln_final
        self.text_projection = clip_model.text_projection
        self.dtype = clip_model.dtype
        self.token_embedding = clip_model.token_embedding

    def forward(self, text):
        x = self.token_embedding(text).type(self.dtype)  # [batch_size, n_ctx, d_model]

        x = x + self.positional_embedding.type(self.dtype)
        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD
        x = self.ln_final(x).type(self.dtype)

        # x.shape = [batch_size, n_ctx, transformer.width]
        # take features from the eot embedding (eot_token is the highest number in each sequence)
        x = x[torch.arange(x.shape[0]), text.argmax(dim=-1)] @ self.text_projection

        return x

def load_clip_to_cpu(visual_backbone):
    backbone_name = visual_backbone
    url = clip._MODELS[backbone_name]
    model_path = clip._download(url, os.path.expanduser("~/.cache/clip"))

    try:
        # loading JIT archive
        model = torch.jit.load(model_path, map_location="cpu").eval()
        state_dict = None

    except RuntimeError:
        state_dict = torch.load(model_path, map_location="cpu")

    model = clip.build_model(state_dict or model.state_dict())

    return model

clip_model = load_clip_to_cpu("RN50")

visual_model = torch.nn.DataParallel(clip_model.visual).cuda()

text_model = TextEncoder(clip_model)
text_model = torch.nn.DataParallel(text_model).cuda()

Image Encodings

In [61]:
# Encode images as CLIP embeddings: Unnormalized
output_path_images = f"/nethome/bdevnani3/flash1/long_tail_lang/datasets/ImageNet_emb/RN50/images/"
for inp, label, index, path in tqdm(dl):
    out = visual_model(inp.half()).float()
    new_path = path[0].split("ImageNet")[1]
    new_path =new_path.replace(".JPEG", ".pt")
#     new_path =new_path.replace("train", split)
    new_path = output_path_images+new_path
    new_path = pathlib.Path(new_path)
    new_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(out, new_path)

  0%|          | 0/115846 [00:00<?, ?it/s]

Text Encodings

In [6]:
# Encode labels as CLIP embeddings: Unnormalized
output_path = "/nethome/bdevnani3/flash1/long_tail_lang/datasets/ImageNet_emb/RN50/labels"


# Save prompt paths
prompt_indices = {}
for i, prompt in enumerate(classes.GENERIC_PROMPT_COLLECTIONS["ImageNet"]):
    prompt_indices[i] = prompt
fp = pathlib.Path(output_path)
fp.mkdir(exist_ok=True)
with open(output_path+'/prompt_indices.json', 'w') as f:
    json.dump(prompt_indices, f)


In [9]:
# Save embeddings
for c, actual_label in tqdm(enumerate(classes.CLASSES)):
    for i, prompt in enumerate(classes.GENERIC_PROMPT_COLLECTIONS["ImageNet"]):

        text = clip.tokenize(prompt.format(actual_label))
        texts = text.cuda()
        text_embedding = text_model(texts).float()

        new_path = output_path+f"/{c}/{i}.pt"
        new_path = pathlib.Path(new_path)
        new_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(text_embedding, new_path)
    

0it [00:00, ?it/s]

In [55]:
#save image path to label mapping
fp = pathlib.Path(output_path+f'/{split}/image_to_label.txt')
fp.parent.mkdir(parents=True, exist_ok=True)
with open(fp, 'w') as f:
    for inp, label, index, path in tqdm(dl):
        new_path = path[0].split("ImageNet")[1]
        new_path =new_path.replace(".JPEG", ".pt")
#         new_path =new_path.replace("train", split)
        new_path = output_path_images+new_path
        new_path = pathlib.Path(new_path)

        actual_label = classes.CLASSES[int(label)]
        label = label.item()
        onlyfiles = [str(f.absolute()) for f in pathlib.Path(f"{output_path}/{label}/").glob("*.pt")]
        to_write = [str(new_path), str(label), "_".join(str(actual_label).split(" "))]
        to_write.extend(onlyfiles)
        to_write.append("\n")
        
        f.write(" ".join(to_write))

  0%|          | 0/115846 [00:00<?, ?it/s]